### Olist Required Metadata & Silver Transactional Tables



In [ ]:
# The tables below are required metadata and some silver transactional tables
# that need to be created beforehand for data to be loaded succesfully

# Developer: Asif Shah
# Created:   14.09.2025

In [ ]:
%%sql

--Create the metadata control tables if they doesn't exist already 
--This is required to ensure deduplication of pipeline activities and file data
--as well as to allow for metadata driven pipelines

--This cell has been frozen and made inactive as its only really required on 
--first run 


CREATE TABLE IF NOT EXISTS dbo.audit_control
(
folder_name STRING,
file_name STRING,
file_rows INTEGER,
table_name STRING,
table_rows INTEGER,
status STRING, 
processed_timestamp TIMESTAMP
);


CREATE TABLE IF NOT EXISTS dbo.metadata_control
(
Filename STRING,
Tablename STRING
);

CREATE TABLE IF NOT EXISTS dbo.last_rows_check
(
folder_name STRING,
file_rows INTEGER,
table_rows INTEGER,
row_diff   INTEGER
);


CREATE TABLE IF NOT EXISTS dbo.unprocessed_folders
(
folder_name STRING,
file_name STRING,
table_name STRING
);


In [ ]:
# Creating a blank silver order items table using pyspark rather than spark.sql method


from pyspark.sql.types import StringType, StructField, StructType, IntegerType, DoubleType, TimestampType


schema = StructType ([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True ),
    StructField("seller_id", StringType(), True ),
    StructField("shipping_limit_date", TimestampType() , True ),
    StructField("price", DoubleType(), True ),
    StructField("freight_value", DoubleType(), True ),
    StructField("source_folder", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("silver_ingested_at", TimestampType(), True)

])

empty_order_item_table = spark.createDataFrame([] ,schema)

empty_order_item_table.write.format("delta").mode("ignore").saveAsTable("Silver.olist_order_items")

In [ ]:
# Creating a blank silver payments table using pyspark rather than spark.sql method


from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# 1. Map out the target schema layout
payment_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True),
    StructField("silver_ingested_at", TimestampType(), True)
])

# 2. Generate a totally empty dataframe using a blank list
blank_df = spark.createDataFrame([], schema=payment_schema)

# 3. Commit the empty structural design directly to the catalogue
blank_df.write.format("delta").mode("ignore").saveAsTable("Silver.olist_order_payments")


In [ ]:
# Creating the olist states mapping table that wil enrich the data in the
# silver layer by adding the actual state name against the state initials

from pyspark.sql.types import StructType, StructField, StringType

data = [
 ("AC", "Acre"), 
 ("AL", "Alagoas"), 
 ("AP", "Amapá"), 
 ("AM", "Amazonas"),
 ("BA", "Bahia"), 
 ("CE", "Ceará"), 
 ("DF", "Distrito Federal"), 
 ("ES", "Espírito Santo"),
 ("GO", "Goiás"), 
 ("MA", "Maranhão"), 
 ("MT", "Mato Grosso"), 
 ("MS", "Mato Grosso do Sul"),
 ("MG", "Minas Gerais"),
 ("PA", "Pará"), 
 ("PB", "Paraíba"), 
 ("PR", "Paraná"),
 ("PE", "Pernambuco"), 
 ("PI", "Piauí"), 
 ("RJ", "Rio de Janeiro"), 
 ("RN", "Rio Grande do Norte"),
 ("RS", "Rio Grande do Sul"), 
 ("RO", "Rondônia"), 
 ("RR", "Roraima"), 
 ("SC", "Santa Catarina"),
 ("SP", "São Paulo"), 
 ("SE", "Sergipe"), 
 ("TO", "Tocantins")
]

schema = StructType([
    StructField("state_code", StringType(), False),
    StructField("state_name", StringType(), False),
])

df2_states = spark.createDataFrame(data, schema = schema)

df2_states.write.format("delta").mode("overwrite").saveAsTable("dbo.olist_states_mapping")
# df_check = spark.read.table("dbo.olist_states_mapping")
# df_check.show()